# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 217MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=8bfa6e22-062b-46ef-94e6-98c73ec5c023
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 55.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 191MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta.uniform_(-eps, eps)
    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp(min=1e-12) * eps
    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp(min=1e-12)
        delta = delta * (eps / l2_norm(delta).clamp(min=1e-12)).clamp(max=1)
        delta = clip_box(x, delta)

    return (x + delta).detach()


In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  union 48.4   (40 s)


{'clean': 78.7, 'linf': 48.4, 'union': 48.4}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model


def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def linf_loss(model, x, y):
    model.eval()
    x_adv = attack(model, x, y, "linf")
    model.train()
    return F.cross_entropy(model(x_adv), y)

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
model_standard = adv_train(make_model(), standard_loss, epochs=5)
model_linf = adv_train(make_model(), linf_loss, epochs=5)
model_p3_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")

epoch 1: loss 1.877 (5 s)
epoch 2: loss 1.540 (11 s)
epoch 3: loss 1.383 (16 s)
epoch 4: loss 1.235 (22 s)
epoch 5: loss 1.148 (28 s)
epoch 1: loss 2.192 (47 s)
epoch 2: loss 2.107 (91 s)
epoch 3: loss 2.062 (137 s)
epoch 4: loss 2.041 (182 s)
epoch 5: loss 2.025 (227 s)
epoch 1: loss 1.203 (45 s)
epoch 2: loss 1.180 (90 s)
epoch 3: loss 1.152 (136 s)
epoch 4: loss 1.126 (181 s)
epoch 5: loss 1.096 (226 s)
epoch 6: loss 1.068 (271 s)
epoch 7: loss 1.049 (316 s)
epoch 8: loss 1.030 (361 s)
epoch 9: loss 1.013 (406 s)
epoch 10: loss 1.007 (451 s)


In [ ]:
model_p3_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_p3_l2)

epoch 1: loss 0.893 (47 s)
epoch 2: loss 0.828 (91 s)
epoch 3: loss 0.785 (136 s)
epoch 4: loss 0.743 (181 s)
epoch 5: loss 0.704 (227 s)
epoch 6: loss 0.663 (272 s)
epoch 7: loss 0.633 (317 s)
epoch 8: loss 0.606 (362 s)
epoch 9: loss 0.583 (407 s)
epoch 10: loss 0.572 (452 s)
clean 83.0  linf 32.5  l2 59.9  l1 28.7  union 24.9   (137 s)


{'clean': 83.0, 'linf': 32.5, 'l2': 59.9, 'l1': 28.7, 'union': 24.9}

In [ ]:
test_model(model_standard)
test_model(model_linf)
test_model(model_p3_linf)

clean 57.9  linf 0.3  l2 9.9  l1 1.8  union 0.3   (140 s)
clean 36.3  linf 23.1  l2 29.3  l1 20.9  union 19.3   (137 s)
clean 78.5  linf 45.1  l2 56.9  l1 20.0  union 19.9   (137 s)


{'clean': 78.5, 'linf': 45.1, 'l2': 56.9, 'l1': 20.0, 'union': 19.9}

In [ ]:
model_p3_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_p3_l1)

epoch 1: loss 1.232 (47 s)
epoch 2: loss 0.902 (91 s)
epoch 3: loss 0.614 (137 s)
epoch 4: loss 0.493 (182 s)
epoch 5: loss 0.315 (227 s)
epoch 6: loss 0.239 (272 s)
epoch 7: loss 0.222 (317 s)
epoch 8: loss 0.186 (362 s)
epoch 9: loss 0.142 (408 s)
epoch 10: loss 0.145 (453 s)
clean 83.2  linf 0.0  l2 0.0  l1 27.1  union 0.0   (137 s)


{'clean': 83.2, 'linf': 0.0, 'l2': 0.0, 'l1': 27.1, 'union': 0.0}

In [ ]:
torch.save(model_p3_linf.state_dict(), "model_p3_linf.pt")
torch.save(model_p3_l2.state_dict(), "model_p3_l2.pt")
torch.save(model_p3_l1.state_dict(), "model_p3_l1.pt")

In [ ]:
for eps in [2/255, 4/255, 8/255, 16/255]:
    print(f"\nEpsilon: {eps:.4f}")

    for name, model in [("Standard", model_standard), ("Adversarial", model_linf)]:
        model.eval()
        correct_fgsm = 0
        correct_pgd = 0

        for x, y in batches(x_test, y_test, 250, False):
            x_fgsm = pgd_linf(model, x, y, eps, eps, 1, False)
            x_pgd = pgd_linf(model, x, y, eps, eps / 4, 20)

            with torch.no_grad():
                correct_fgsm += (model(x_fgsm).argmax(1) == y).sum().item()
                correct_pgd += (model(x_pgd).argmax(1) == y).sum().item()

        print(f"{name}: FGSM {correct_fgsm / len(y_test) * 100:.1f}%, PGD {correct_pgd / len(y_test) * 100:.1f}%")


Epsilon: 0.0078
Standard: FGSM 28.5%, PGD 23.4%
Adversarial: FGSM 33.1%, PGD 33.1%

Epsilon: 0.0157
Standard: FGSM 12.7%, PGD 6.5%
Adversarial: FGSM 29.8%, PGD 29.7%

Epsilon: 0.0314
Standard: FGSM 3.6%, PGD 0.3%
Adversarial: FGSM 23.8%, PGD 23.4%

Epsilon: 0.0627
Standard: FGSM 0.6%, PGD 0.0%
Adversarial: FGSM 15.3%, PGD 13.0%


In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    loss_q = F.cross_entropy(logits_q, y, reduction="none")
    loss_r = F.cross_entropy(logits_r, y, reduction="none")
    loss = torch.maximum(loss_q, loss_r).mean()

    mask = logits_q.argmax(1) == y
    if mask.any():
        p_q = F.softmax(logits_q[mask], dim=1)
        log_p_r = F.log_softmax(logits_r[mask], dim=1)
        kl = F.kl_div(log_p_r, p_q, reduction="batchmean")
        loss = loss + lam * kl

    return loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    gn = g_n.flatten()
    ga = g_a.flatten()
    cos = F.cosine_similarity(gn, ga, dim=0, eps=1e-12)
    return torch.clamp(cos, min=0) * g_n

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (97 s)
epoch 2: loss 1.330 (193 s)
epoch 3: loss 1.303 (289 s)
epoch 4: loss 1.281 (385 s)
epoch 5: loss 1.254 (481 s)
epoch 6: loss 1.227 (577 s)
epoch 7: loss 1.199 (673 s)
epoch 8: loss 1.186 (769 s)
epoch 9: loss 1.171 (865 s)
epoch 10: loss 1.166 (961 s)
clean 79.3  linf 41.0  l2 62.1  l1 44.5  union 39.0   (137 s)


{'clean': 79.3, 'linf': 41.0, 'l2': 62.1, 'l1': 44.5, 'union': 39.0}

In [ ]:
torch.save(model_ramp.state_dict(), "model_ramp.pt")

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [ ]:
def my_loss(model, x, y):
    # TODO: your method
    model.eval()
    examples = [attack(model, x, y, norm) for norm in ("linf", "l2", "l1")]
    model.train()

    losses = [F.cross_entropy(model(x_adv), y, reduction="none") for x_adv in examples]

    with torch.no_grad():
        accuracies = [(model(x_adv).argmax(1) == y).float().mean() for x_adv in examples]
        weakest = torch.stack(accuracies).argmin().item()

    weights = [1.0, 1.0, 1.0]
    weights[weakest] = 2.0

    return sum(w * loss.mean() for w, loss in zip(weights, losses)) / sum(weights)


In [ ]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.233 (137 s)
epoch 2: loss 1.146 (285 s)
epoch 3: loss 1.116 (434 s)
epoch 4: loss 1.086 (584 s)
epoch 5: loss 1.051 (733 s)
epoch 6: loss 1.028 (883 s)
epoch 7: loss 1.001 (1032 s)
epoch 8: loss 0.984 (1181 s)
epoch 9: loss 0.974 (1331 s)
epoch 10: loss 0.970 (1481 s)


NameError: name 'test_model' is not defined

In [ ]:
torch.save(model_mine.state_dict(), "model_mine.pt")

In [ ]:
test_model(model_mine)

clean 78.2  linf 41.9  l2 63.3  l1 45.9  union 39.9   (146 s)


{'clean': 78.2, 'linf': 41.9, 'l2': 63.3, 'l1': 45.9, 'union': 39.9}

In [ ]:
model_mine_seed1 = adv_train(load_base_model(), my_loss, seed=1)
torch.save(model_mine_seed1.state_dict(), "model_mine_seed1.pt")
test_model(model_mine_seed1)

epoch 1: loss 1.230 (122 s)
epoch 2: loss 1.151 (245 s)
epoch 3: loss 1.118 (369 s)
epoch 4: loss 1.090 (494 s)
epoch 5: loss 1.061 (619 s)
epoch 6: loss 1.028 (744 s)
epoch 7: loss 1.000 (869 s)
epoch 8: loss 0.985 (993 s)
epoch 9: loss 0.971 (1118 s)
epoch 10: loss 0.958 (1242 s)
clean 78.6  linf 42.0  l2 62.3  l1 44.8  union 39.1   (118 s)


{'clean': 78.6, 'linf': 42.0, 'l2': 62.3, 'l1': 44.8, 'union': 39.1}

In [ ]:
model_ramp_seed1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
torch.save(model_ramp_seed1.state_dict(), "model_ramp_seed1.pt")
test_model(model_ramp_seed1)

epoch 1: loss 1.434 (81 s)
epoch 2: loss 1.332 (165 s)
epoch 3: loss 1.298 (249 s)
epoch 4: loss 1.274 (333 s)
epoch 5: loss 1.247 (418 s)
epoch 6: loss 1.221 (504 s)
epoch 7: loss 1.200 (589 s)
epoch 8: loss 1.186 (674 s)
epoch 9: loss 1.179 (759 s)
epoch 10: loss 1.168 (844 s)
clean 79.1  linf 42.4  l2 62.8  l1 46.4  union 40.6   (118 s)


{'clean': 79.1, 'linf': 42.4, 'l2': 62.8, 'l1': 46.4, 'union': 40.6}

In [ ]:
test_model(model_mine_seed1, steps=200, restarts=5)

clean 78.6  linf 41.9  l2 62.2  l1 43.5  union 38.6   (1166 s)


{'clean': 78.6, 'linf': 41.9, 'l2': 62.2, 'l1': 43.5, 'union': 38.6}

In [ ]:
model_ramp_15 = adv_train(load_base_model(), ramp_loss, epochs=15, gp_beta=0.5, seed=1)
torch.save(model_ramp_15.state_dict(), "model_ramp_15.pt")
test_model(model_ramp_15)

epoch 1: loss 1.434 (82 s)
epoch 2: loss 1.333 (165 s)
epoch 3: loss 1.302 (249 s)
epoch 4: loss 1.279 (334 s)
epoch 5: loss 1.255 (419 s)
epoch 6: loss 1.229 (504 s)
epoch 7: loss 1.209 (589 s)
epoch 8: loss 1.191 (674 s)
epoch 9: loss 1.174 (759 s)
epoch 10: loss 1.150 (844 s)
epoch 11: loss 1.137 (929 s)
epoch 12: loss 1.114 (1014 s)
epoch 13: loss 1.107 (1099 s)
epoch 14: loss 1.106 (1185 s)
epoch 15: loss 1.101 (1270 s)
clean 79.7  linf 40.9  l2 60.5  l1 45.0  union 39.1   (118 s)


{'clean': 79.7, 'linf': 40.9, 'l2': 60.5, 'l1': 45.0, 'union': 39.1}

In [ ]:
def my_loss_ablation(model, x, y):
    model.eval()
    examples = [attack(model, x, y, norm) for norm in ("linf", "l2", "l1")]
    model.train()

    losses = [F.cross_entropy(model(x_adv), y) for x_adv in examples]
    return sum(losses) / 3

In [ ]:
model_ablation = adv_train(load_base_model(), my_loss_ablation, seed=0)
torch.save(model_ablation.state_dict(), "model_ablation.pt")
test_model(model_ablation)

epoch 1: loss 1.194 (118 s)
epoch 2: loss 1.113 (239 s)
epoch 3: loss 1.079 (360 s)
epoch 4: loss 1.046 (482 s)
epoch 5: loss 1.008 (603 s)
epoch 6: loss 0.986 (724 s)
epoch 7: loss 0.956 (845 s)
epoch 8: loss 0.938 (966 s)
epoch 9: loss 0.927 (1087 s)
epoch 10: loss 0.923 (1209 s)
clean 79.1  linf 40.2  l2 62.8  l1 47.3  union 38.6   (121 s)


{'clean': 79.1, 'linf': 40.2, 'l2': 62.8, 'l1': 47.3, 'union': 38.6}